[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Migrations with Alembic &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, writes the models to
`scratch/college_models.py`, and makes the migration environment and its baseline, as the first two
worked examples did. Run it first. The tasks follow one another, as a project's revisions do, and
the last cell removes the scratch folder.


In [1]:
import logging
import os
import re
import shlex
import shutil
import subprocess
import sys
from datetime import date
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("alembic")
except PackageNotFoundError:                                        # Colab has no Alembic: install the version this notebook runs
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", "alembic==1.20.0"],
                   check=True)

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, event, func, insert,
                        select, text)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))
print("alembic", version("alembic"))


PROJECT = Path("scratch")
MODELS = PROJECT / "college_models.py"
VERSIONS = PROJECT / "migrations" / "versions"


def alembic(*arguments):
    """Run an alembic command in the project folder, and print what it printed, less the lines every command repeats."""
    engine.dispose()                                                # the notebook's own connections close first
    done = subprocess.run(
        [sys.executable, "-m", "alembic", *arguments], cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env={**os.environ, "NO_COLOR": "1", "PYTHONDONTWRITEBYTECODE": "1", "PYTHONUNBUFFERED": "1"},
    )
    lines = done.stdout.replace(f"{PROJECT.resolve()}{os.sep}", "").splitlines()
    if "Traceback (most recent call last):" in lines:              # the error's last line, not Python's own files
        lines = lines[:lines.index("Traceback (most recent call last):")] + ["Traceback (most recent call last): ...", lines[-1]]
    print("$", shlex.join(["alembic", *arguments]))
    for line in lines:
        if not any(noise in line for noise in ("Context impl", "Will assume", "setting up autogenerate plugin")):
            print("   ", line)


def edit(path, old, new):
    """Replace the one place in a file where old appears with new."""
    source = Path(path).read_text()
    assert source.count(old) == 1, f"{old!r} appears {source.count(old)} times in {path}"
    Path(path).write_text(source.replace(old, new))


MODELS.write_text(r'''"""The college's tables, as classes: the models Alembic compares the database with."""
from datetime import date

from sqlalchemy import CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"
''')

alembic("init", "migrations")

ini = PROJECT / "alembic.ini"
ini.write_text(re.sub(r"^sqlalchemy\.url = .*$", "sqlalchemy.url = sqlite:///college.db", ini.read_text(), flags=re.M))
edit(PROJECT / "migrations" / "env.py", "target_metadata = None", "from college_models import Base\n\ntarget_metadata = Base.metadata")


def revision_file(rev_id):
    """The file of the revision with this id."""
    return next(VERSIONS.glob(f"{rev_id}_*.py"))


def print_revision(rev_id):
    """Print a revision's upgrade and downgrade, leaving out its header, which holds the time it was written."""
    source = revision_file(rev_id).read_text()
    print(source[source.index("def upgrade"):].rstrip())


alembic("revision", "--autogenerate", "-m", "baseline", "--rev-id", "0001")
print_revision("0001")
alembic("stamp", "head")
alembic("current")


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}
alembic 1.20.0
$ alembic init migrations
    Creating directory migrations ...  done
    Creating directory migrations/versions ...  done
    Generating migrations/script.py.mako ...  done
    Generating migrations/env.py ...  done
    Generating migrations/README ...  done
    Generating alembic.ini ...  done
    Please edit configuration/connection/logging settings in alembic.ini before proceeding.
$ alembic revision --autogenerate -m baseline --rev-id 0001
    Generating migrations/versions/0001_baseline.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    pass
    # ### end Alembic commands ###


def downgrade() -> None:
    """Downgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    pass
    # ### end Alembic commands ###
$ alembic stamp head
    INFO

**1.** A new column on `courses`, from the model to the database.


In [2]:
edit(MODELS, "    credits: Mapped[int]\n",
     "    credits: Mapped[int]\n    description: Mapped[str | None] = mapped_column(String(500))\n")
alembic("revision", "--autogenerate", "-m", "add course description", "--rev-id", "0002")
print_revision("0002")
alembic("upgrade", "head")
print([column["name"] for column in sqlalchemy.inspect(engine).get_columns("courses")])


$ alembic revision --autogenerate -m 'add course description' --rev-id 0002
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'courses.description'
    Generating migrations/versions/0002_add_course_description.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.add_column('courses', sa.Column('description', sa.String(length=500), nullable=True))
    # ### end Alembic commands ###


def downgrade() -> None:
    """Downgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.drop_column('courses', 'description')
    # ### end Alembic commands ###
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, add course description
['id', 'code', 'title', 'department', 'credits', 'description']


`String(500)` became `VARCHAR(500)`, and `str | None` made the column one that may be empty,
`nullable=True`.


**2.** The revision undone, and the project as it was.


In [3]:
alembic("downgrade", "-1")
revision_file("0002").unlink()
edit(MODELS, "    description: Mapped[str | None] = mapped_column(String(500))\n", "")
alembic("current")
print([column["name"] for column in sqlalchemy.inspect(engine).get_columns("courses")])


$ alembic downgrade -1
    INFO  [alembic.runtime.migration] Running downgrade 0002 -> 0001, add course description
$ alembic current
    0001 (head)
['id', 'code', 'title', 'department', 'credits']


The downgrade dropped the column, and removing the file and the model's line leaves nothing for a
later autogenerate to find again.


**3.** A revision by hand, which adds a row.


In [4]:
alembic("revision", "-m", "add fall 2026", "--rev-id", "0003")
edit(revision_file("0003"), '"""Upgrade schema."""\n    pass',
     '"""Upgrade schema."""\n    op.execute("INSERT INTO terms (name, starts_on) VALUES (\'Fall 2026\', \'2026-08-24\')")')
edit(revision_file("0003"), '"""Downgrade schema."""\n    pass',
     '"""Downgrade schema."""\n    op.execute("DELETE FROM terms WHERE name = \'Fall 2026\'")')
print_revision("0003")
alembic("upgrade", "head")
with engine.connect() as conn:
    print(conn.execute(select(func.count()).select_from(Term)).scalar(), "terms")


$ alembic revision -m 'add fall 2026' --rev-id 0003
    Generating migrations/versions/0003_add_fall_2026.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    op.execute("INSERT INTO terms (name, starts_on) VALUES ('Fall 2026', '2026-08-24')")


def downgrade() -> None:
    """Downgrade schema."""
    op.execute("DELETE FROM terms WHERE name = 'Fall 2026'")
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0003, add fall 2026
5 terms


A revision can change data as well as tables. Autogenerate never writes one like this, since the
models say nothing about rows.


**4.** The SQL of that revision.


In [5]:
alembic("upgrade", "0001:0003", "--sql")


$ alembic upgrade 0001:0003 --sql
    INFO  [alembic.runtime.migration] Generating static SQL
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0003, add fall 2026
    -- Running upgrade 0001 -> 0003
    
    INSERT INTO terms (name, starts_on) VALUES ('Fall 2026', '2026-08-24');
    
    UPDATE alembic_version SET version_num='0003' WHERE alembic_version.version_num = '0001';
    


The `INSERT` is written out as it is in the revision, followed by the `UPDATE` of `alembic_version`.


**5.** Rooms, in batch mode.


In [6]:
edit(PROJECT / "migrations" / "env.py", "connection=connection, target_metadata=target_metadata",
     "connection=connection, target_metadata=target_metadata, render_as_batch=True")
edit(MODELS, "class Section(Base):", """class Room(Base):
    __tablename__ = "rooms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))


class Section(Base):""")
edit(MODELS, "    capacity: Mapped[int]\n",
     "    capacity: Mapped[int]\n    room_id: Mapped[int | None] = mapped_column(ForeignKey(\"rooms.id\"))\n")
alembic("revision", "--autogenerate", "-m", "add rooms", "--rev-id", "0004")
print_revision("0004")
alembic("upgrade", "head")
print([key["name"] for key in sqlalchemy.inspect(engine).get_foreign_keys("sections")])


$ alembic revision --autogenerate -m 'add rooms' --rev-id 0004
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'rooms'
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'sections.room_id'
    INFO  [alembic.autogenerate.compare.constraints] Detected added foreign key (room_id)(id) on table sections
    Generating migrations/versions/0004_add_rooms.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.create_table('rooms',
    sa.Column('id', sa.Integer(), nullable=False),
    sa.Column('name', sa.String(length=50), nullable=False),
    sa.PrimaryKeyConstraint('id', name=op.f('pk_rooms'))
    )
    with op.batch_alter_table('sections', schema=None) as batch_op:
        batch_op.add_column(sa.Column('room_id', sa.Integer(), nullable=True))
        batch_op.create_foreign_key(batch_op.f('fk_sections_room_id_rooms'), 'rooms', ['room_id'], ['id'])

    # ### end Alembic 

`render_as_batch=True` first, since `sections` exists and SQLite cannot add a foreign key to it in
place. The copy kept the two foreign keys `sections` had and added the new one.


**6.** Down to the baseline, and up again.


In [7]:
alembic("downgrade", "0001")
alembic("current")
alembic("upgrade", "head")
alembic("current")


$ alembic downgrade 0001
    INFO  [alembic.runtime.migration] Running downgrade 0004 -> 0003, add rooms
    INFO  [alembic.runtime.migration] Running downgrade 0003 -> 0001, add fall 2026
$ alembic current
    0001
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0003, add fall 2026
    INFO  [alembic.runtime.migration] Running upgrade 0003 -> 0004, add rooms
$ alembic current
    0004 (head)


`downgrade 0001` ran the `downgrade()` of every revision after 0001, newest first, and `upgrade head`
ran them all again.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Migrations with Alembic](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/17-migrations-with-alembic.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
